# Phase 1 — Interpretability Harness Validation

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/PatrickJReed/cerberus-neuro/blob/main/notebooks/04_phase_1_harness.ipynb)

**Goal:** Run the full Phase 1 interpretability harness against the existing `cerberus-neuro-v0-baseline` checkpoint (val_acc 0.73). Validate every method end-to-end and produce inputs for the Phase 1 results report.

**Spec:** `docs/superpowers/specs/2026-05-12-argus-cells-design.md`, §4 + §5 + §6.
**Plan:** `docs/superpowers/plans/2026-05-12-argus-cells-phase-1-interpretability-harness.md`.
**Predecessor:** Phase 0 audit (PROCEED, 48 donors, crop budget ≫ target).

In [ ]:
!pip install -q --upgrade git+https://github.com/PatrickJReed/cerberus-neuro.git@main

In [ ]:
from huggingface_hub import login
from google.colab import drive, userdata

login(userdata.get("HF_TOKEN"))
drive.mount("/content/drive")

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

from cerberus_neuro.data import build_manifest, subset_manifest, well_level_split, NeuroPaintingDataset
from cerberus_neuro.model import BaselineDiseaseClassifier
from cerberus_neuro.attribution import (
    compute_channel_ablation,
    compute_gradcam,
    compute_integrated_gradients,
)
from cerberus_neuro.probes import parallel_probe_report
from cerberus_neuro.analysis import (
    plot_channel_ablation_heatmap,
    plot_probe_comparison,
    stratify_channel_scores_by_cell_type,
)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"device: {DEVICE}")
if DEVICE == "cuda":
    print(f"gpu: {torch.cuda.get_device_name(0)}")

CACHE_DIR = Path("/content/drive/MyDrive/cerberus-neuro/cache")
CACHE_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
from huggingface_hub import hf_hub_download

# Phase 1's harness validates against the existing 0.73 baseline (best-epoch:
# 12 or 13 per docs/superpowers/results/2026-05-08-v0-phase-1-baseline-result.md).
HF_REPO = "patrickjreed/cerberus-neuro-v0-baseline"
CKPT_FILE = "epoch_013.pt"  # best-epoch val_acc 0.7311

ckpt_path = hf_hub_download(HF_REPO, CKPT_FILE)
ckpt = torch.load(ckpt_path, map_location="cpu", weights_only=False)

model = BaselineDiseaseClassifier(in_channels=6, n_classes=2, pretrained_encoder=False)
model.load_state_dict(ckpt["model"])
model = model.to(DEVICE).eval()
print(f"Loaded {HF_REPO}/{CKPT_FILE}")
print(f"Checkpoint epoch={ckpt.get('epoch')}, step={ckpt.get('step')}")
print(f"Parameter count: {model.parameter_count()}")